# Fe(SH)4-: ORCA CASSCF(5e,5o) and FCIDUMP

Two ORCA calculations:

1. A high-spin ROHF, whose five singly occupied orbitals (Fe 3d) are the starting orbitals for the active space.
2. A CASSCF(5e,5o) for each spin state, which writes the active-space integrals to an FCIDUMP file.

The FCIDUMP files and a JSON summary (spin, energy, wall time) are saved in `orca output/`.

## Imports and paths

In [1]:
import os, ctypes, subprocess
import cudaq

print("GPUs seen by CUDA-Q:", cudaq.num_available_gpus())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("nvidia-smi:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True))

# Load every GPU library CUDA-Q wants, and print the real reason if one fails.
for lib in os.environ["CUDAQ_DYNLIBS"].split(":"):
    try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
    except OSError as err:
        print("FAILED:", err)

cudaq.set_target("nvidia", option="fp32")
print("GPU target OK")

GPUs seen by CUDA-Q: 1
CUDA_VISIBLE_DEVICES: None
nvidia-smi: CompletedProcess(args=['nvidia-smi', '-L'], returncode=0, stdout='GPU 0: Tesla T4 (UUID: GPU-8c11fc28-c414-4062-78a2-7663c09cddb8)\n', stderr='')
GPU target OK


In [2]:
import os
import shutil
import time
from pathlib import Path

import pandas as pd

from opi.core import Calculator
from opi.input.blocks import BlockCasscf, BlockOutput
from opi.input.simple_keywords import BasisSet, Scf, ShellType
from opi.input.structures import Structure

In [3]:
# ORCA.
orca_dir = os.path.expanduser("~/Software/orca-7.0.0-f.beta.4_linux_x86-64/bin")
os.environ["PATH"] = orca_dir + os.pathsep + os.environ["PATH"]

# Open MPI 4.1.8, installed in the virtual environment with `pip install openmpi==4.1.8`.
mpi_dir = os.path.expanduser("~/.venvs/quiche-cudaq")
os.environ["OPI_MPI"] = mpi_dir
os.environ["PATH"] = mpi_dir + "/bin" + os.pathsep + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = mpi_dir + "/lib" + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

# Allow 4 MPI processes inside a cluster (PBS) job.
os.environ["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

# CudaQ CPU backend
os.environ["OMP_NUM_THREADS"] = "8"

## Settings

In [4]:
# Spin states: 2 (low spin), 4 (intermediate spin) and 6 (high spin).
multiplicities = [2, 4, 6]

# Active space: the five Fe 3d orbitals and their five electrons.
nel, norb = 5, 5

# Parallelisation: one MPI process per physical core, memory in MB per process.
ncores = 4
memory = 6000

# Directories: the geometry is in FeSH4/, each calculation gets its own subdirectory.
workdir = Path.cwd() / "FeSH4"
rohf_dir = workdir / "rohf"
rohf_dir.mkdir(parents=True, exist_ok=True)

# The FCIDUMP files and the JSON summary go to demo/orca output/.
output_dir = Path.cwd() / "orca_output"
output_dir.mkdir(exist_ok=True)

## Step 1: High-spin ROHF (starting orbitals)

In [5]:
# Set up the ORCA calculator.
calc = Calculator(basename="FeSH4_rohf", working_dir=rohf_dir)
calc.structure = Structure.from_xyz(workdir / "fesh4.xyz")
calc.structure.charge, calc.structure.multiplicity = -1, 6
calc.input.ncores = ncores
calc.input.memory = memory
calc.input.add_simple_keywords(ShellType.ROHF, BasisSet.DEF2_SVP)

# Run the calculation.
calc.write_input()
if not (rohf_dir / "FeSH4_rohf.gbw").exists(): calc.run()

# Extract the output.
output = calc.get_output()
if not output.terminated_normally():
    raise RuntimeError("ORCA did not terminate normally.")
output.parse()
print("ROHF energy:", output.get_final_energy())

ROHF energy: -2854.3781932168554


/data/home/xenov/.venvs/quiche-cudaq/lib/python3.12/site-packages/opi/output/core.py:533: UserWarning: Could not determine ORCA version from JSON file. Subsequent error might occur.
  warn("Could not determine ORCA version from JSON file. Subsequent error might occur.")


## Step 2: CASSCF(5e,5o) and FCIDUMP for each spin state

In [6]:
# Check backend options of orca

results = pd.read_json(output_dir / "orca_results.json").to_dict("records") if (output_dir / "orca_results.json").exists() else []

for multiplicity in [m for m in multiplicities if not (output_dir / f"FeSH4_mult{m}.fcidump").exists()]:
    casscf_dir = workdir / f"casscf_mult{multiplicity}"
    casscf_dir.mkdir(parents=True, exist_ok=True)

    # Set up the ORCA calculator, starting from the ROHF orbitals.
    calc = Calculator(basename="FeSH4", working_dir=casscf_dir)
    calc.structure = Structure.from_xyz(workdir / "fesh4.xyz")
    calc.structure.charge, calc.structure.multiplicity = -1, multiplicity
    calc.input.ncores = ncores
    calc.input.memory = memory
    calc.input.add_simple_keywords(ShellType.ROHF, BasisSet.DEF2_SVP, Scf.MOREAD)
    shutil.copy(rohf_dir / "FeSH4_rohf.gbw", casscf_dir / "FeSH4_rohf.gbw")
    calc.input.moinp = casscf_dir / "FeSH4_rohf.gbw"

    # Set up the active space.
    calc.input.add_blocks(BlockCasscf(nel=nel, norb=norb, mult=multiplicity))

    # Set up saving the integrals.
    output_block = BlockOutput()
    output_block.add_option("DumpActInts", "True")
    calc.input.add_blocks(output_block)

    # Run the calculation and time it.
    calc.write_input()
    start = time.perf_counter()
    calc.run()
    walltime = time.perf_counter() - start

    # Extract the output.
    output = calc.get_output()
    if not output.terminated_normally():
        raise RuntimeError(f"ORCA did not terminate normally for multiplicity {multiplicity}.")
    output.parse()

    # Save the FCIDUMP file and the results for this spin state.
    shutil.copy(casscf_dir / "FeSH4.fcidump", output_dir / f"FeSH4_mult{multiplicity}.fcidump")
    results.append({
        "multiplicity": multiplicity,
        "spin": (multiplicity - 1) / 2,
        "energy": output.get_final_energy(),
        "walltime_s": walltime,
    })
    print(f"Multiplicity {multiplicity}: E = {output.get_final_energy():.8f} Ha in {walltime:.0f} s")

In [7]:
# Save the summary of all spin states.
df = pd.DataFrame(results)
df.to_json(output_dir / "orca_results.json", orient="records", indent=2)
df

,multiplicity,spin,energy,walltime_s
0,2,0.5,-2854.213658,822.842428
1,4,1.5,-2854.269899,314.308871
2,6,2.5,-2854.378194,85.519572


## Step 3: DMRG state preparation

In [8]:
import json
import subprocess

# Julia (installed with juliaup).
os.environ["PATH"] = os.path.expanduser("~/.juliaup/bin") + os.pathsep + os.environ["PATH"]

In [9]:
import math

# High spin (multiplicity 6) is a single determinant: one spin-up electron in each of the five d orbitals.
# MPSCircuits cannot compile this product state (it needs 0 layers), so we write its circuit directly:
# Ry(pi) flips |0> to |1> on each spin-up qubit (targets 1, 3, 5, 7, 9 in the file's 1-based numbering).
circuit = {"num_qubits": 10, "operations": [{"op": "Ry", "angle": math.pi, "target": q} for q in (1, 3, 5, 7, 9)]}
(output_dir / "FeSH4_mult6_dmrg_circuit.json").write_text(json.dumps(circuit, indent=2))

# For a single determinant the exact energy is the CASSCF energy; <S^2> = S(S+1) = 8.75.
e_high = next(r["energy"] for r in results if r["multiplicity"] == 6)
info = {"dmrg_energy": e_high, "s2": 8.75, "refined_circuit_fidelity": 1.0}
(output_dir / "FeSH4_mult6_dmrg_info.json").write_text(json.dumps(info, indent=2))

86

In [10]:
dmrg_info = {m: json.loads((output_dir / f"FeSH4_mult{m}_dmrg_info.json").read_text()) for m in multiplicities if (output_dir / f"FeSH4_mult{m}_dmrg_info.json").exists()}

for multiplicity in [m for m in multiplicities if not (output_dir / f"FeSH4_mult{m}_dmrg_info.json").exists()]:
    print(f"Calculating for spin multiplicity {multiplicity}.")
    fcidump_path = output_dir / f"FeSH4_mult{multiplicity}.fcidump"
    circuit_path = output_dir / f"FeSH4_mult{multiplicity}_dmrg_circuit.json"

    # DMRG in the target spin state (2S = multiplicity - 1), compiled into a circuit.
    out = subprocess.run(
        ["julia", "+1.12", "--project=dmrg_env", "dmrg_circuit.jl", str(fcidump_path), str(circuit_path),
         f"--ms2={multiplicity - 1}", "--spin_penalty=1.0"],
        capture_output=True, text=True, check=True,
    ).stdout
    print(out)

    # Save the DMRG summary (energy, <S^2>, fidelity) next to the circuit.
    dmrg_info[multiplicity] = dict(line.split("=") for line in out.splitlines() if "=" in line)
    (output_dir / f"FeSH4_mult{multiplicity}_dmrg_info.json").write_text(json.dumps(dmrg_info[multiplicity], indent=2))

In [11]:
from statePrep.cudaq_circuit_loading import load_circuit_from_json, write_and_import_kernel

dmrg = {}
for multiplicity in multiplicities:
    circuit_path = output_dir / f"FeSH4_mult{multiplicity}_dmrg_circuit.json"
    # A different kernel name per spin state, so the three kernels don't overwrite each other.
    dmrg[multiplicity] = write_and_import_kernel(
        load_circuit_from_json(circuit_path), kernel_name=f"dmrg_mult{multiplicity}", module_name=f"dmrg_mult{multiplicity}"
    )

In [12]:
from cudaq_algorithms import chemistry
import cudaq
from quiche.chemistry import HartreeFockState, get_jw_state
from quiche.io import fcidump

num_qubits = 2 * norb
num_electrons = nel

# Jordan-Wigner qubit Hamiltonian for each spin state, from its FCIDUMP.
h_cq = {}
for multiplicity in multiplicities:
    sq_ham = fcidump.read(str(output_dir / f"FeSH4_mult{multiplicity}.fcidump"))
    h_cq[multiplicity] = chemistry.qubit_hamiltonian(sq_ham.one_body, sq_ham.two_body, scalar_offset=sq_ham.core_energy)

In [13]:
def make_dmrg_state(dmrg_kernel):
    """Wrap a DMRG circuit in a kernel that allocates its own qubits (one kernel per spin state)."""

    @cudaq.kernel
    def dmrg_state():
        q = cudaq.qvector(num_qubits)
        dmrg_kernel(q)

    return dmrg_state


# CASSCF over the whole active space, i.e. FCI within the active orbitals.
fci_energy = {r["multiplicity"]: r["energy"] for r in results}

checks = []
for multiplicity in multiplicities:
    dmrg_state = make_dmrg_state(dmrg[multiplicity])

    # Reference determinant for this spin state: the lowest orbitals filled with
    # n_alpha spin-up electrons (even qubits) and n_beta spin-down electrons (odd qubits).
    n_alpha = (num_electrons + multiplicity - 1) // 2
    n_beta = num_electrons - n_alpha
    occupation = [0] * num_qubits
    for i in range(n_alpha):
        occupation[2 * i] = 1
    for i in range(n_beta):
        occupation[2 * i + 1] = 1
    hf = HartreeFockState(occupation=tuple(occupation))
    hf_bits = "".join(str(int(b)) for b in get_jw_state(hf.occupation))

    energy = cudaq.observe(dmrg_state, h_cq[multiplicity]).expectation()
    overlap = abs(cudaq.get_state(dmrg_state).amplitude(hf_bits)) ** 2
    print(f"Multiplicity {multiplicity}: DMRG energy via CUDA-Q: {energy:.8f}   FCI: {fci_energy[multiplicity]:.8f}   |<HF|DMRG>|^2: {overlap:.4f}")

    checks.append({
        "multiplicity": multiplicity,
        "energy_cudaq": energy,
        "energy_dmrg": float(dmrg_info[multiplicity]["dmrg_energy"]),
        "energy_fci": fci_energy[multiplicity],
        "hf_overlap": overlap,
    })

# Save the check for all spin states.
pd.DataFrame(checks).to_json(output_dir / "dmrg_check.json", orient="records", indent=2)

Multiplicity 2: DMRG energy via CUDA-Q: -2854.17441733   FCI: -2854.21365762   |<HF|DMRG>|^2: 0.9266
Multiplicity 4: DMRG energy via CUDA-Q: -2854.24689887   FCI: -2854.26989948   |<HF|DMRG>|^2: 0.9694
Multiplicity 6: DMRG energy via CUDA-Q: -2854.37819435   FCI: -2854.37819435   |<HF|DMRG>|^2: 1.0000


## Step 4: Set up and run the QPE comparison

In [14]:
import numpy as np

import quiche
from quiche.core import ElectronicHamiltonian, Errors, PhaseEstimation, Simulation
from quiche.cudaq import measured_circuit_kernel, sample_post_processor, textbook_qpe_circuit
from quiche.dispatch import QPESpec

# QPE evolves H - c0, and c0 is added back to every measured energy.
# c0 (nuclear repulsion + frozen core) is thousands of Ha. Left in, it dominates the 1-norm that sets the
# evolution time t = pi/lambda, and the readout grid becomes far too coarse to see the active-space energy.
h_qpe, c0 = {}, {}
for multiplicity in multiplicities:
    h = quiche.PauliSum.from_cudaq(h_cq[multiplicity])
    c0[multiplicity], h_qpe[multiplicity] = h.split_identity()
    print(f"Multiplicity {multiplicity}: {h.num_words} Pauli terms, lambda = {h.lam:.1f} Ha, without c0: {h_qpe[multiplicity].lam:.1f} Ha")

Multiplicity 2: 875 Pauli terms, lambda = 2864.4 Ha, without c0: 11.4 Ha
Multiplicity 4: 743 Pauli terms, lambda = 2864.4 Ha, without c0: 11.3 Ha
Multiplicity 6: 487 Pauli terms, lambda = 2864.2 Ha, without c0: 11.1 Ha


In [15]:
def get_qpe(multiplicity, error, reps=None):
    # Split the budget between the Trotter error and the QPE error; QPESpec picks the
    # QPE ancillas and the Trotter time/order/steps from it.
    qpe = QPESpec(
        hamiltonian=ElectronicHamiltonian(
            num_electrons=num_electrons, mapping=quiche.Mapping.JordanWigner, paulis=h_qpe[multiplicity]
        ),
        state_prep=None,  # the DMRG kernel is injected below instead
        algorithm=PhaseEstimation.Textbook,
        simulation=Simulation.Trotter,
        error_budget=Errors(
            estimation=error / 2,
            simulation=error / 2,
            rotations=error / 2,
            state_prep=error / 2,
            overlap=1.0,
        ),
    )
    # Default trotter step budgetting if no custom is given.
    if reps is None:
            reps = qpe.reps 

    kernel, work_qubits, decode = textbook_qpe_circuit(
        h_qpe[multiplicity], Simulation.Trotter, qpe.num_qpe_ancillas, qpe.time, reps, qpe.order,
        n_qubits=qpe.num_data,
    )
    circuit = measured_circuit_kernel(
        kernel, work_qubits, qpe.num_qpe_ancillas, state_prep=dmrg[multiplicity], num_data=qpe.num_data
    )
    post = sample_post_processor(decode, 1)
    return qpe, circuit, post

In [ ]:
# Different output file if we limit trotter budgeting. None = default quiche qpe.
trotter_reps = 1
results_file = output_dir / f"qpe_trotter_reps_{trotter_reps}_results.json"

shots = 1000
chem_acc = 1.6e-3  # Ha

# Upper limit on the phase register per backend; each extra phase qubit roughly doubles the runtime.
max_phase_qubits = {"GPU": 15, "CPU": 15}

qpe_results = pd.read_json(results_file).to_dict("records") if (results_file).exists() else []
done = {(r["backend"], r["multiplicity"], r["error_budget"]): r for r in qpe_results}

for backend in ["GPU", "CPU"]:
    # Select the backend: the GPU in single precision, the CPU in double precision.
    if backend == "GPU":
        cudaq.set_target("nvidia", option="fp32")
    else:
        cudaq.set_target("qpp-cpu")
    cudaq.sample(make_dmrg_state(dmrg[multiplicities[0]]), shots_count=1)  # start the backend up outside the timing

    for multiplicity in multiplicities:
        # Halve the error budget (one extra phase qubit each time) until we reach chemical accuracy.
        error = 2.0
        while True:
            if (backend, multiplicity, error) in done:
                result = done[(backend, multiplicity, error)]  # already run earlier: reuse it
            else:
                qpe, circuit, post = get_qpe(multiplicity, error, reps=trotter_reps)
                if qpe.num_qpe_ancillas > max_phase_qubits[backend]:
                    print(f"{backend} | mult {multiplicity} | stopped: would need more than {max_phase_qubits[backend]} phase qubits")
                    break

                # Simulate and time it.
                start = time.perf_counter()
                samples = cudaq.sample(circuit, shots_count=shots)
                walltime = time.perf_counter() - start

                energy = float(np.median(post(samples))) + c0[multiplicity]
                result = {
                    "backend": backend,
                    "multiplicity": multiplicity,
                    "error_budget": error,
                    "phase_qubits": qpe.num_qpe_ancillas,
                    "total_qubits": qpe.num_qpe_ancillas + qpe.num_data,
                    "trotter_reps": trotter_reps or qpe.reps,
                    "energy": energy,
                    "abs_error": abs(energy - fci_energy[multiplicity]),
                    "walltime_s": walltime,
                }
                # Save after every run, so an interrupted run loses nothing.
                qpe_results.append(result)
                pd.DataFrame(qpe_results).to_json(results_file, orient="records", indent=2)
                print(f"{backend} | mult {multiplicity} | {result['phase_qubits']} phase qubits | "
                      f"|E - E_FCI| = {result['abs_error']:.2e} Ha | {walltime:.1f} s")

            if result["abs_error"] <= chem_acc:
                print(f"{backend} | mult {multiplicity} | chemical accuracy reached with {result['phase_qubits']} phase qubits")
                break
            error /= 2

GPU | mult 2 | 4 phase qubits | |E - E_FCI| = 2.26e-01 Ha | 3.7 s
GPU | mult 2 | 5 phase qubits | |E - E_FCI| = 2.26e-01 Ha | 4.1 s
GPU | mult 2 | 6 phase qubits | |E - E_FCI| = 1.30e-01 Ha | 5.1 s
GPU | mult 2 | 7 phase qubits | |E - E_FCI| = 4.76e-02 Ha | 6.7 s
GPU | mult 2 | 8 phase qubits | |E - E_FCI| = 4.14e-02 Ha | 9.9 s
GPU | mult 2 | 9 phase qubits | |E - E_FCI| = 3.08e-03 Ha | 15.8 s
GPU | mult 2 | 10 phase qubits | |E - E_FCI| = 3.08e-03 Ha | 56.5 s
GPU | mult 2 | 11 phase qubits | |E - E_FCI| = 3.08e-03 Ha | 287.9 s
GPU | mult 2 | 12 phase qubits | |E - E_FCI| = 2.49e-03 Ha | 1072.1 s
GPU | mult 2 | 13 phase qubits | |E - E_FCI| = 2.93e-04 Ha | 4159.5 s
GPU | mult 2 | chemical accuracy reached with 13 phase qubits
GPU | mult 4 | 4 phase qubits | |E - E_FCI| = 2.50e-01 Ha | 2.8 s
GPU | mult 4 | 5 phase qubits | |E - E_FCI| = 2.50e-01 Ha | 3.3 s
GPU | mult 4 | 6 phase qubits | |E - E_FCI| = 1.02e-01 Ha | 4.0 s
GPU | mult 4 | 7 phase qubits | |E - E_FCI| = 7.39e-02 Ha | 5.4 s
